# 04. Polynomial Regression

After preprocessing, expand features with `PolynomialFeatures`, then classify with
linear regression plus a 0.5 threshold. Choose `degree` by stratified CV (F1);
`degree=3` is tried but may be dropped if it is too slow or memory-heavy.

In [1]:
import sys
sys.path.append("..")

from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures

from src.data import load_adult
from src.preprocess import split_features_target, make_model_pipeline
from src.evaluate import (
    cross_val_scores,
    evaluate_on_test,
    save_results,
    ThresholdingRegressor, 
)

X, y = split_features_target(load_adult("train"))
X_test, y_test = split_features_target(load_adult("test"))
print(X.shape, y.mean())

(32561, 12) 0.2408095574460244


In [2]:
# Find the best degree
candidates = [2, 3]
selection_metric = "f1"

cv_by_degree = {}
best_degree = None
best_score = -1.0

for degree in candidates:
    poly_reg = Pipeline([
        ("poly", PolynomialFeatures(degree=degree, include_bias=False)),
        ('reg', LinearRegression())
    ])

    model = ThresholdingRegressor(poly_reg, threshold=0.5)
    pipe = make_model_pipeline(model)

    cv = cross_val_scores(pipe, X, y)
    cv_by_degree[degree] = cv
    score = cv[selection_metric]["mean"]
    print(f"degree={degree} CV {selection_metric}={score:.4f} (±{cv[selection_metric]['std']:.4f})")

    if score > best_score:
        best_score = score
        best_degree = degree

print("selected degree:", best_degree)


degree=2 CV f1=0.6594 (±0.0070)


KeyboardInterrupt: 

In [3]:
poly_reg = Pipeline([
    ("poly", PolynomialFeatures(degree=best_degree, include_bias=False)),
    ("reg", LinearRegression()),
])

final_model = ThresholdingRegressor(poly_reg, threshold=0.5)

cv = cv_by_degree[best_degree]
pipe2 = make_model_pipeline(final_model)

test = evaluate_on_test(pipe2, X, y, X_test, y_test)

path = save_results(
    f"polynomial_regression_deg{best_degree}",
    cv,
    test,
    extra={
        "degree": best_degree, 
        "threshold": 0.5,
        "candidates": candidates,
        "selection_metric": selection_metric,
        "cv_by_degree": {
            str(d): {m: v["mean"] for m, v in summary.items()}
            for d, summary in cv_by_degree.items()
        },
    },
)

print("Test:", test["metrics"])
print("Confusion matrix:", test["confusion_matrix"])
print("saved:", path)

Test: {'accuracy': 0.8524660647380382, 'balanced_accuracy': 0.7624378750783325, 'f1': 0.6545872878918608, 'roc_auc': 0.896753257343804, 'average_precision': 0.7461847746999042}
Confusion matrix: [[11603, 832], [1570, 2276]]
saved: /Users/kang/Desktop/ML project/Estimating-Income/results/polynomial_regression_deg2.json
